In [1]:
import pybamm # loads pybamm package
import matplotlib.pyplot as plt # package for plotting
import numpy as np # for arrays
import pandas as pd # for structure use .csv for importing and exporting
import math # log, sin, exp
from scipy.integrate import solve_ivp # integration, used in accelerated simulation
import pickle # for saving simulations

%matplotlib widget

In [2]:
parameter_values = pybamm.ParameterValues(chemistry=pybamm.parameter_sets.Siegel2022)
spme = pybamm.lithium_ion.SPMe()
param = spme.param

In [3]:
parameter_values.search("maximum concentration")

Maximum concentration in negative electrode [mol.m-3]	28746.0
Maximum concentration in positive electrode [mol.m-3]	35380.0


## Setup Electrode stoichoimetries and capacities here for model initialization

In [4]:
# Cn
C_n_init = 4.19
# Cp
C_p_init = 3.85
# x0 or x100 depending on starting at 0 SOC or 1 SOC
x_init = 0.005
# y0 or y100 depending on starting at 0 SOC or 1 SOC
y_init = 0.92

$e^-_s=\frac{C_n*3600}{F l^- c^-_{s,max} A}$

where $l^-$ is thickness of anode, $c^-_{s,max}$ is maximum concentration and $A$ is area of electrode

In [5]:
eps_n_init = parameter_values.evaluate(C_n_init*3600/(param.n.L * param.n.prim.c_max * param.F* param.A_cc))
eps_p_init = parameter_values.evaluate(C_p_init*3600/(param.p.L * param.p.prim.c_max * param.F* param.A_cc))

$x_{init}=x_0\, or\, x_{100} \times c^-_{s,max}$
$y_{init}=y_0\, or\, y_{100} \times c^+_{s,max}$

In [6]:
cs_n_init = parameter_values.evaluate(x_init* param.n.prim.c_max)
cs_p_init = parameter_values.evaluate(y_init* param.p.prim.c_max)

In [7]:
# Updating parameters for initialization
parameter_values.update(
    {
        "Negative electrode active material volume fraction": eps_n_init,
        "Positive electrode active material volume fraction": eps_p_init,
        "Initial concentration in negative electrode [mol.m-3]":cs_n_init,
        "Initial concentration in positive electrode [mol.m-3]":cs_p_init,
        "Initial temperature [K]": 273.15+25,
        "Ambient temperature [K]": 273.15+25,
        "Lower voltage cut-off [V]":2.7,
        "Upper voltage cut-off [V]":4.2,
    },
    check_already_exists=False,
)

In [8]:
c_rate_d = "C/3"
c_rate_c = "C/3"


experiment = pybamm.Experiment(
    [
        ("Discharge at "+c_rate_d+" until 3V",
        "Rest for 10 sec",
        "Charge at "+c_rate_c+" until 4.2V", 
        "Hold at 4.2V until C/20")
    ] ,
    # ] *40,
    termination="50% capacity",
#     cccv_handling="ode",
)

sim_0 = pybamm.Simulation(spme, experiment=experiment, parameter_values=parameter_values, 
                            solver=pybamm.CasadiSolver("safe"))
sol_0 = sim_0.solve()

t = sol_0["Time [s]"].entries
I = sol_0["Current [A]"].entries
Q = sol_0['Discharge capacity [A.h]'].entries
Vt = sol_0["Terminal voltage [V]"].entries

CasADi - 2023-09-29 12:19:49 WARNING("The options 't0', 'tf', 'grid' and 'output_t0' have been deprecated.
The same functionality is provided by providing additional input arguments to the 'integrator' function, in particular:
 * Call integrator(..., t0, tf, options) for a single output time, or
 * Call integrator(..., t0, grid, options) for multiple grid points.
The legacy 'output_t0' option can be emulated by including or excluding 't0' in 'grid'.
Backwards compatibility is provided in this release only.") [.../casadi/core/integrator.cpp:515]
